In [ ]:
!pip install openai gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [14]:
import os
from openai import OpenAI
from gtts import gTTS
from IPython.display import Audio, display
from google.colab import userdata

# ==============================
# 🔑 CONFIGURAÇÃO INICIAL
# ==============================

# Chave implementada diretamente conforme solicitado
api_key = userdata.get('OPENAI_API_KEY') # Retrieve API key securely from Colab secrets
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

print("✅ Chave configurada e cliente inicializado!")

# ==============================
# 📝 SPEECH-TO-TEXT (WHISPER)
# ==============================

def transcrever_audio(caminho_audio):
    try:
        with open(caminho_audio, "rb") as audio_file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )
        texto = transcript.text
        print(f"Transcrição: {texto}")
        return texto
    except Exception as e:
        print("Erro no Whisper:", e)
        return None

# ==============================
# 🤖 CHATGPT (INTELIGÊNCIA MULTI-IDIOMA)
# ==============================

def gerar_resposta(texto_usuario):
    try:
        resposta = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Você é um assistente multilíngue. Responda sempre no mesmo idioma em que o usuário falar com você."},
                {"role": "user", "content": texto_usuario}
            ]
        )
        texto_resposta = resposta.choices[0].message.content
        print("ChatGPT:", texto_resposta)
        return texto_resposta
    except Exception as e:
        print("Erro no ChatGPT:", e)
        return None

# ==============================
# 🔊 TEXT-TO-SPEECH (gTTS)
# ==============================

def falar_resposta(texto, idioma_detectado='pt'):
    try:
        tts = gTTS(text=texto, lang=idioma_detectasdo)
        tts.save("resposta.mp3")
        display(Audio("resposta.mp3", autoplay=True))
    except Exception as e:
        print("Erro no gTTS:", e)

print("Funções prontas para uso.")

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [ ]:
from google.colab import output
import base64

# JavaScript para gravar áudio no navegador
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def capturar_audio_colab(segundos=5):
    print(f"Ouvindo por {segundos} segundos...")
    display(output.eval_js(RECORD_JS))
    audio_b64 = output.eval_js(f"record({segundos*1000})")
    audio_bytes = base64.b64decode(audio_b64.split(',')[1])

    caminho_arquivo = "input_audio.wav"
    with open(caminho_arquivo, "wb") as f:
        f.write(audio_bytes)

    return caminho_arquivo

In [ ]:
def loop_conversa():
    """
    Executa um turno de conversa: Grava -> Transcreve -> Responde -> Fala
    """
    # 1. Captura
    arquivo_audio = capturar_audio_colab(5)

    # 2. STT (Whisper)
    texto_usuario = transcrever_audio(arquivo_audio)

    if texto_usuario:
        # 3. ChatGPT
        resposta = gerar_resposta(texto_usuario)

        if resposta:
            # 4. TTS (gTTS)
            falar_resposta(resposta)

In [15]:
from google.colab import output, userdata
import base64
import os
from openai import OpenAI
from gtts import gTTS
from IPython.display import Audio, display

# 1. Configuração do Cliente
# Certifique-se de que a variável de ambiente OPENAI_API_KEY está configurada
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY')) # Retrieve API key securely from Colab secrets

# 2. JavaScript para Gravação
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def capturar_audio(segundos=5):
    print(f"🎤 Ouvindo por {segundos} segundos...")
    display(output.eval_js(RECORD_JS))
    audio_b64 = output.eval_js(f"record({segundos*1000})")
    audio_bytes = base64.b64decode(audio_b64.split(',')[1])
    with open("input.wav", "wb") as f:
        f.write(audio_bytes)
    return "input.wav"

def processar_conversa():
    try:
        # Captura
        path = capturar_audio(5)

        # Transcrição (Whisper)
        with open(path, "rb") as f:
            transcript = client.audio.transcriptions.create(model="whisper-1", file=f)
        print(f"Você: {transcript.text}")

        # Resposta (GPT)
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": "Responda de forma curta no idioma do usuário."},
                      {"role": "user", "content": transcript.text}]
        )
        texto_res = res.choices[0].message.content
        print(f"ChatGPT: {texto_res}")

        # Voz (gTTS)
        tts = gTTS(text=texto_res, lang='pt')
        tts.save("res.mp3")
        display(Audio("res.mp3", autoplay=True))

    except Exception as e:
        print(f"Erro: {e}")

print("✅ Funções definidas com sucesso! Agora você pode chamar 'processar_conversa()'.")

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [ ]:
import os

# Certifique-se de que a API Key está configurada no ambiente
# Se houver erro de API Key, execute a célula de configuração inicial novamente.

print("🎙️ Iniciando nova conversa...")
try:
    processar_conversa()
except NameError:
    print("❌ Erro: As funções não foram definidas. Por favor, execute a célula anterior (ebcca0c6) primeiro.")
except Exception as e:
    print(f"❌ Ocorreu um erro: {e}")

🎙️ Iniciando nova conversa...
🎤 Ouvindo por 5 segundos...


None

Erro: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [ ]:
print("🎙️ TESTE INICIADO - Aguardando áudio...")
try:
    processar_conversa()
except Exception as e:
    print(f"Ocorreu um erro no teste: {e}")

🎙️ TESTE INICIADO - Aguardando áudio...
🎤 Ouvindo por 5 segundos...


None

Erro: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [ ]:
import os
from IPython.display import Audio, display

# Teste de gravação e reprodução imediata
arquivo = capturar_audio_colab(5)

print("Reproduzindo o áudio capturado para verificação...")
if os.path.exists(arquivo):
    display(Audio(arquivo, autoplay=True))
else:
    print("Erro: Arquivo de áudio não foi gerado.")

🎤 Ouvindo por 5 segundos...


None

Reproduzindo o áudio capturado para verificação...
